In [1]:
!nvidia-smi


Fri Oct  2 08:11:04 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   45C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [6]:
%cd 3-day-inference-challenge

/kaggle/working/3-day-inference-challenge


In [7]:
!bash scripts/kaggle_bootstrap.sh

index, name, memory.total [MiB], driver_version
0, Tesla T4, 15360 MiB, 580.178.04
1, Tesla T4, 15360 MiB, 580.178.04
	GPU0	GPU1	CPU Affinity	NUMA Affinity	GPU NUMA ID
GPU0	 X 	PHB	0-3	0		N/A
GPU1	PHB	 X 	0-3	0		N/A

Legend:

  X    = Self
  SYS  = Connection traversing PCIe as well as the SMP interconnect between NUMA nodes (e.g., QPI/UPI)
  NODE = Connection traversing PCIe as well as the interconnect between PCIe Host Bridges within a NUMA node
  PHB  = Connection traversing PCIe as well as a PCIe Host Bridge (typically the CPU)
  PXB  = Connection traversing multiple PCIe bridges (without traversing the PCIe Host Bridge)
  PIX  = Connection traversing at most a single PCIe bridge
  NV#  = Connection traversing a bonded set of # NVLinks
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for infer-lab (pyproje

In [8]:
!infer-lab bench --config configs/smoke.yaml --tag smoke


2026-10-02 08:16:09,615 INFO numexpr.utils: NumExpr defaulting to 4 threads.
2026-10-02 08:16:19,216 INFO infer_lab.bench.runner: loading Qwen/Qwen2.5-0.5B-Instruct on hf_torch
2026-10-02 08:16:19,531 INFO httpx: HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-10-02 08:16:19,532 WARNING huggingface_hub.utils._http: Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-10-02 08:16:19,538 INFO httpx: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-0.5B-Instruct/7ae557604adf67be50417f59c2c2f167def9a775/config.json "HTTP/1.1 200 OK"
2026-10-02 08:16:19,545 INFO httpx: HTTP Request: GET https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-0.5B-Instruct/7ae557604adf67be50417f59c2c2f167def9a775/config.json "HTTP/1.1 200 OK"
config.json: 100%|█████████████████████████████| 659/659 [0

In [10]:
!infer-lab bench --config configs/bench.yaml --tag baseline


2026-10-02 08:21:14,361 INFO numexpr.utils: NumExpr defaulting to 4 threads.
2026-10-02 08:21:21,579 INFO infer_lab.bench.runner: loading Qwen/Qwen2.5-0.5B-Instruct on hf_torch
2026-10-02 08:21:21,864 INFO httpx: HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-10-02 08:21:21,870 INFO httpx: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-0.5B-Instruct/7ae557604adf67be50417f59c2c2f167def9a775/config.json "HTTP/1.1 200 OK"
2026-10-02 08:21:22,100 INFO httpx: HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-10-02 08:21:22,100 WARNING huggingface_hub.utils._http: Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-10-02 08:21:22,105 INFO httpx: HTTP Request: HEAD https://huggingface.co/api/res

In [11]:
!infer-lab kernels


vector_add  torch         n=67108864     3.281 ms    245.5 GB/s  ( 76.7% of T4 peak)
vector_add  scalar        n=67108864     3.570 ms    225.6 GB/s  ( 70.5% of T4 peak)
vector_add  vec4          n=67108864     3.385 ms    237.9 GB/s  ( 74.4% of T4 peak)
matmul      torch_cublas  n=512          0.067 ms   4.033 TFLOPS  ( 49.8% of T4 peak)
matmul      naive         n=512          0.449 ms   0.598 TFLOPS  (  7.4% of T4 peak)
matmul      tiled16       n=512          0.293 ms   0.917 TFLOPS  ( 11.3% of T4 peak)
matmul      torch_cublas  n=1024         0.363 ms   5.910 TFLOPS  ( 73.0% of T4 peak)
matmul      naive         n=1024         4.990 ms   0.430 TFLOPS  (  5.3% of T4 peak)
matmul      tiled16       n=1024         2.933 ms   0.732 TFLOPS  (  9.0% of T4 peak)
matmul      torch_cublas  n=2048         4.836 ms   3.553 TFLOPS  ( 43.9% of T4 peak)
matmul      naive         n=2048        42.591 ms   0.403 TFLOPS  (  5.0% of T4 peak)
matmul      tiled16       n=2048        28.274 ms   0.608

In [12]:
!infer-lab report


2026-10-02 08:24:50,250 INFO numexpr.utils: NumExpr defaulting to 4 threads.
 id      tag  backend  batch_size  prompt_len  output_len  output_tokens_per_s  ttft_p50_ms  ttft_p99_ms  tpot_p50_ms  e2e_p99_ms
  1    smoke hf_torch           1          64          32                34.25        37.17        37.17        28.92      933.59
  2    smoke hf_torch           4          64          32               132.33        31.79        31.79        30.15      966.56
  3 baseline hf_torch           1         128         128                33.76        33.36        33.66        29.44     3842.15
  4 baseline hf_torch           1         512         128                33.79        46.67        47.11        29.59     3826.54
  5 baseline hf_torch           4         128         128               137.84        32.15        33.54        29.09     3736.30
  6 baseline hf_torch           4         512         128               132.42       175.42       175.77        28.96     3909.17
  7 baseline 

: 